In [1]:
# 11 GLM v3 + claim history, GBM + history, blend: two-fold validation
# Purpose: add the claim-history features (src/history.py) and measure the gain on the same folds.
# Rules fixed before results:
#  (a) history is adopted in the GLM if average validation D2 rises by >= 0.0005 and the A/E of rows
#      with a past claim moves toward 1;
#  (b) GBM or blend is adopted only if >= 0.003 above GLM v3 + history, with lowest/highest decile A/E
#      no more than 0.1 further from 1 than the GLM's.
import sys, time
sys.path.append("..")
import numpy as np
import pandas as pd
import lightgbm as lgb
from src.target import LINES, load_policies, load_claims, build_model_target
from src.features import build_features
from src.history import claim_history
from src.glm import BandedTweedieGLM
from src.evaluate import evaluate
from src.diagnostics import FOLDS

NON_FEATURES = ("row_id", "year", "exposure_clean", "target", "weight")
DROP = {"motor": ["previous_insurer", "customer_tenure"],
        "buildings": ["previous_insurer", "occupation", "region", "customer_tenure"],
        "contents": ["previous_insurer", "occupation", "customer_tenure"]}
HIST = ["hist_1y", "hist_2y"]
GLM_CONFIG = {"motor": dict(alpha=0.5, n_bands=10, drop=None, balance=True),
              "buildings": dict(alpha=0.5, n_bands=20, drop=None, balance=True),
              "contents": dict(alpha=0.3, n_bands=20, drop=None, balance=True)}
PARAMS2 = dict(objective="tweedie", tweedie_variance_power=1.5, metric="tweedie", learning_rate=0.03,
               num_leaves=31, min_data_in_leaf=500, feature_fraction=0.8, bagging_fraction=0.8,
               bagging_freq=1, lambda_l2=1.0, min_data_per_group=1000, cat_smooth=50, cat_l2=10,
               max_cat_threshold=8, verbose=-1, seed=42)


def hist_levels(n):
    """Counts -> '0', '1', '2+' ; unknown history stays missing (its own level in the GLM)."""
    return pd.Series(np.select([n == 0, n == 1, n >= 2], ["0", "1", "2+"], default=None), index=n.index,
                     dtype="object").where(n.notna())


def add_history(F, p_rows, p_all_same_line, claims):
    h = claim_history(p_rows, p_all_same_line, claims)
    h["hist_1y"] = hist_levels(h.pol_claims_1y)
    h["hist_2y"] = hist_levels(h.pol_claims_2y)
    out = F.merge(h[["row_id"] + HIST], on="row_id", how="left", validate="one_to_one")
    assert len(out) == len(F)
    return out


def prep2(X, cats=None):
    X = X.copy()
    if cats is None:
        cats = {c: sorted(X[c].dropna().astype(str).unique()) for c in X.columns
                if not pd.api.types.is_numeric_dtype(X[c]) and not pd.api.types.is_bool_dtype(X[c])}
    for c, levels in cats.items():
        s = X[c].astype("string")
        X[c] = pd.Categorical(s.where(s.isin(levels)), categories=levels)
    for c in X.columns:
        if pd.api.types.is_bool_dtype(X[c]):
            X[c] = X[c].astype(int)
    return X, cats


def fit_gbm2(tr, feats, params=PARAMS2, patience=300):
    last = tr.year.max()
    inner, hold = tr[tr.year < last], tr[tr.year == last]
    Xi, cats = prep2(inner[feats]); Xh, _ = prep2(hold[feats], cats)
    di = lgb.Dataset(Xi, inner.target / inner.weight, weight=inner.weight)
    dh = lgb.Dataset(Xh, hold.target / hold.weight, weight=hold.weight, reference=di)
    b = lgb.train(params, di, num_boost_round=5000, valid_sets=[dh],
                  callbacks=[lgb.early_stopping(patience, verbose=False)])
    n_trees = max(int(b.best_iteration * 1.1), 1)
    Xt, cats = prep2(tr[feats])
    model = lgb.train(params, lgb.Dataset(Xt, tr.target / tr.weight, weight=tr.weight), num_boost_round=n_trees)
    k = tr.target.sum() / (model.predict(Xt) * tr.weight).sum()
    return model, cats, k, n_trees


# 1. Modelling tables with history (train); test history uses train + test periods of the same line
claims = load_claims()
data = {}
for line in LINES:
    p, pt = load_policies(line, "train"), load_policies(line, "test")
    mt, comp, loading = build_model_target(claims, p, line)
    F = add_history(build_features(line, "train", policies=p), p, [p, pt], claims)
    df = mt.merge(F, on="row_id", how="left", validate="one_to_one")
    data[line] = {"df": df, "loading": loading}
    print(f"{line}: hist_1y shares by year\n",
          df.groupby("year").hist_1y.value_counts(normalize=True, dropna=False).unstack().round(3).to_string())

# 2. Two-fold validation
rows, hist_ae = [], []
for line in LINES:
    df = data[line]["df"]
    base = [c for c in df.columns if c not in NON_FEATURES and c not in DROP[line] and c not in HIST]
    withh = base + HIST
    for vy, tys in FOLDS:
        t0 = time.time()
        tr, va = df[df.year.isin(tys)], df[df.year == vy]
        out = {}
        for name, feats in [("GLM v3", base), ("GLM v3 + history", withh)]:
            glm = BandedTweedieGLM(**GLM_CONFIG[line]).fit(tr[feats], tr.target / tr.weight, tr.weight)
            out[name] = glm.predict_rate(va[feats]) * va.weight.to_numpy()
        model, cats, k, n_trees = fit_gbm2(tr, withh)
        Xv, _ = prep2(va[withh], cats)
        out["GBM + history"] = k * model.predict(Xv) * va.weight.to_numpy()
        out["blend + history"] = 0.5 * out["GLM v3 + history"] + 0.5 * out["GBM + history"]
        for name, pred in out.items():
            r = evaluate(va.target, pred, va.weight, show=False)
            lf = r["lift"]["A/E"]
            rows.append({"line": line, "year": vy, "model": name, "D2": r["D2"], "A/E": r["A/E"],
                         "low": lf.iloc[0], "top": lf.iloc[-1]})
            g = pd.DataFrame({"h": va.hist_1y.fillna("unknown").values, "a": va.target.values, "p": pred})
            s = g.groupby("h")[["a", "p"]].sum()
            for lvl, ae in (s.a / s.p).items():
                hist_ae.append({"line": line, "year": vy, "model": name, "hist_1y": lvl, "A/E": ae})
        print(f"{line} {vy}: GBM {n_trees} trees, k = {k:.3f}, {time.time() - t0:.0f}s")

res = pd.DataFrame(rows)
res.to_csv("../results/history_validation.csv", index=False)
for line in LINES:
    t = res[res.line == line].pivot(index="model", columns="year", values=["D2", "A/E", "low", "top"])
    t.columns = [f"{m}_{y}" for m, y in t.columns]
    t["D2_avg"] = t[["D2_2023", "D2_2024"]].mean(axis=1)
    print(f"\n===== {line} =====")
    print(t.sort_values("D2_avg", ascending=False).round(4).to_string())

ha = pd.DataFrame(hist_ae).pivot_table(index=["line", "model"], columns=["hist_1y", "year"], values="A/E")
print("\nA/E by previous-year claims on the same policy (should move toward 1 with history):")
print(ha.round(3).to_string())

avg = res.groupby(["line", "model"]).D2.mean().unstack()
print("\nrule (a): history in the GLM (gain vs GLM v3; adopt if >= 0.0005):")
for line in LINES:
    d = avg.loc[line, "GLM v3 + history"] - avg.loc[line, "GLM v3"]
    print(f"  {line:9s} {d:+.4f}  {'adopt' if d >= 0.0005 else 'do not adopt'}")
print("\nrule (b): GBM / blend vs GLM v3 + history (D2 threshold 0.003; check tails in the tables):")
for line in LINES:
    for m in ["GBM + history", "blend + history"]:
        d = avg.loc[line, m] - avg.loc[line, "GLM v3 + history"]
        print(f"  {line:9s} {m:16s} {d:+.4f}  {'meets D2 threshold' if d >= 0.003 else 'below'}")

motor: hist_1y shares by year
 hist_1y      0      1     2+    NaN
year                               
2021       NaN    NaN    NaN  1.000
2022     0.754  0.084  0.011  0.152
2023     0.755  0.082  0.010  0.153
2024     0.757  0.080  0.009  0.154
buildings: hist_1y shares by year
 hist_1y      0      1     2+    NaN
year                               
2021       NaN    NaN    NaN  1.000
2022     0.815  0.031  0.001  0.153
2023     0.816  0.030  0.001  0.153
2024     0.816  0.031  0.001  0.152
contents: hist_1y shares by year
 hist_1y      0      1     2+    NaN
year                               
2021       NaN    NaN    NaN  1.000
2022     0.817  0.028  0.001  0.154
2023     0.818  0.028  0.001  0.153
2024     0.818  0.029  0.001  0.152
motor 2023: GBM 157 trees, k = 1.089, 46s
motor 2024: GBM 207 trees, k = 1.081, 70s
buildings 2023: GBM 78 trees, k = 1.170, 26s
buildings 2024: GBM 126 trees, k = 1.149, 38s
contents 2023: GBM 82 trees, k = 1.167, 27s
contents 2024: GBM 126 trees, k =

In [2]:
# 11b Final fit on 2021-2024: GLM v4 (= v3 + claim history) and an experimental blend
from pathlib import Path
from src.target import DATA_DIR

subs_glm, subs_blend, checks = [], [], []
for line in LINES:
    df, loading = data[line]["df"], data[line]["loading"]
    feats = [c for c in df.columns if c not in NON_FEATURES and c not in DROP[line]]   # includes HIST
    glm = BandedTweedieGLM(**GLM_CONFIG[line]).fit(df[feats], df.target / df.weight, df.weight)

    p, pt = load_policies(line, "train"), load_policies(line, "test")
    Ft = add_history(build_features(line, "test", policies=pt), pt, [p, pt], claims)
    assert list(Ft.row_id) == list(pt.row_id)
    expo = pt.exposure_clean.to_numpy()
    rate_glm = glm.predict_rate(Ft[feats])
    pred_glm = np.maximum((rate_glm + loading) * expo, 0)
    subs_glm.append(pd.DataFrame({"row_id": Ft.row_id, "prediction": pred_glm}))

    if line == "motor":
        pred_blend = pred_glm
    else:
        model, cats, k, n_trees = fit_gbm2(df, feats)
        Xt, _ = prep2(Ft[feats], cats)
        pred_blend = np.maximum((0.5 * rate_glm + 0.5 * k * model.predict(Xt) + loading) * expo, 0)
    subs_blend.append(pd.DataFrame({"row_id": Ft.row_id, "prediction": pred_blend}))

    d24 = df[df.year == 2024]
    lvl24 = np.average(glm.predict_rate(d24[feats]), weights=d24.exposure_clean)
    checks.append({"line": line, "glm_k": glm.k_,
                   "hist_1y test (0/1/2+/unknown)": Ft.hist_1y.fillna("unk").value_counts(normalize=True)
                       .reindex(["0", "1", "2+", "unk"]).round(3).tolist(),
                   "hist_1y train 2024": d24.hist_1y.fillna("unk").value_counts(normalize=True)
                       .reindex(["0", "1", "2+", "unk"]).round(3).tolist(),
                   "glm test/2024": np.average(rate_glm, weights=expo) / lvl24,
                   "blend/glm total": pred_blend.sum() / pred_glm.sum()})
    print(f"{line}: done")

pd.set_option("display.max_colwidth", 60)
print(pd.DataFrame(checks).set_index("line").to_string())

sample = pd.read_parquet(DATA_DIR / "sample_submission.parquet")
out = Path("../submissions"); out.mkdir(exist_ok=True)
for name, parts in [("glm_v4.csv", subs_glm), ("blend_v2_experiment.csv", subs_blend)]:
    sub = pd.concat(parts, ignore_index=True)
    assert len(sub) == len(sample) and set(sub.row_id) == set(sample.row_id) and sub.row_id.is_unique
    assert sub.prediction.notna().all() and (sub.prediction >= 0).all()
    sub = sample[["row_id"]].merge(sub, on="row_id", how="left")
    sub.to_csv(out / name, index=False)
    print(f"wrote {name}: total {sub.prediction.sum():,.0f}")

motor: done
buildings: done
contents: done
              glm_k hist_1y test (0/1/2+/unknown)            hist_1y train 2024  glm test/2024  blend/glm total
line                                                                                                           
motor      1.019120   [0.759, 0.08, 0.009, 0.152]   [0.757, 0.08, 0.009, 0.154]       0.953806         1.000000
buildings  1.040111  [0.816, 0.031, 0.001, 0.151]  [0.816, 0.031, 0.001, 0.152]       0.985234         0.994027
contents   1.051660  [0.819, 0.029, 0.001, 0.152]  [0.818, 0.029, 0.001, 0.152]       1.012552         0.993051
wrote glm_v4.csv: total 190,249,631
wrote blend_v2_experiment.csv: total 189,705,569


In [3]:
# 11c Unattended experiments. Run AFTER the 11 cell (uses data, add_history, prep2, PARAMS2, GLM_CONFIG, DROP ...).
# Every result is appended to ../results/overnight/ as soon as it exists, so a crash keeps what was done.
#  1. NCB / bonus-hunger check (model-free): do frequency and severity move in opposite directions with NCB?
#  2. Method B: GBM fitted on top of GLM v4 (GLM log-prediction as init_score), default GBM settings.
#  3. GBM settings grid (plain GBM and method B). The setting is chosen on the held-out LAST TRAINING YEAR
#     only; the validation year is used once, for the chosen setting.
# Rules fixed before results (same as before): adopt GBM / method B only if average validation D2 is >= 0.003
# above GLM v4 and lowest/highest decile A/E are no more than 0.1 further from 1 than GLM v4's.
import time, traceback
from pathlib import Path
from itertools import product

OUT = Path("../results/overnight"); OUT.mkdir(parents=True, exist_ok=True)


def log(msg):
    line_ = f"{time.strftime('%H:%M:%S')} {msg}"
    print(line_, flush=True)
    with open(OUT / "log.txt", "a") as f:
        f.write(line_ + "\n")


_ROWS = {}


def append_csv(row, name):
    """Keep all rows in memory and rewrite the whole file, so rows with different columns stay aligned."""
    _ROWS.setdefault(name, []).append(row)
    pd.DataFrame(_ROWS[name]).to_csv(OUT / name, index=False)


def metrics(va, pred):
    r = evaluate(va.target, pred, va.weight, show=False)
    lf = r["lift"]["A/E"]
    return {"D2": r["D2"], "A/E": r["A/E"], "low": lf.iloc[0], "top": lf.iloc[-1]}


def feats_of(df, line):
    return [c for c in df.columns if c not in NON_FEATURES and c not in DROP[line]]


# ---------------------------------------------------------------- 1. NCB check
def ncb_check():
    p = load_policies("motor", "train")
    m = match_claims(claims, p, "motor", cap=None)
    m = m[m.loss_year <= 2023]                                   # mature years only
    df = data["motor"]["df"]
    df = df[df.year <= 2023]
    band = lambda s: pd.cut(s, [-0.5, 0.5, 2.5, 4.5, 8.5, 100], labels=["0", "1-2", "3-4", "5-8", "9+"])
    expo = df.assign(b=band(df.ncb_years)).groupby("b", observed=True).exposure_clean.sum()
    cm = m.merge(df[["row_id", "ncb_years"]], on="row_id", how="inner")
    cm["b"] = band(cm.ncb_years)
    rows = []
    for peril in ["own_damage", "all perils"]:
        s = cm if peril == "all perils" else cm[cm.peril == peril]
        g = s.groupby("b", observed=True)
        t = pd.DataFrame({"freq_per_1000": g.size() / expo * 1000,
                          "mean_gross": g.incurred_to_date.apply(lambda x: np.minimum(x, 500_000).mean()),
                          "small_claim_share": g.apply(lambda x: (x.incurred_to_date <= x.excess + 500).mean()),
                          "exposure_share": expo / expo.sum()})
        t["peril"] = peril
        rows.append(t.reset_index())
    res = pd.concat(rows)
    res.to_csv(OUT / "1_ncb_check.csv", index=False)
    log("1. NCB check (2021-23):\n" + res.round(3).to_string(index=False))


# ------------------------------------------------- shared GLM fits per line and fold
def glm_parts(df, line, vy, tys, feats):
    tr, va = df[df.year.isin(tys)], df[df.year == vy]
    last = tr.year.max()
    inner, hold = tr[tr.year < last], tr[tr.year == last]
    g_in = BandedTweedieGLM(**GLM_CONFIG[line]).fit(inner[feats], inner.target / inner.weight, inner.weight)
    g_tr = BandedTweedieGLM(**GLM_CONFIG[line]).fit(tr[feats], tr.target / tr.weight, tr.weight)
    return dict(tr=tr, va=va, inner=inner, hold=hold,
                li=np.log(g_in.predict_rate(inner[feats])), lh=np.log(g_in.predict_rate(hold[feats])),
                lt=np.log(g_tr.predict_rate(tr[feats])), lv=np.log(g_tr.predict_rate(va[feats])))


def fit_gbm(P, feats, params, offset, patience=300):
    """Plain GBM (offset=False) or method B (offset=True: start from the GLM log-prediction).
    Trees chosen on the held-out last training year; refit on all training years; balance factor k."""
    inner, hold, tr, va = P["inner"], P["hold"], P["tr"], P["va"]
    Xi, cats = prep2(inner[feats]); Xh, _ = prep2(hold[feats], cats)
    di = lgb.Dataset(Xi, inner.target / inner.weight, weight=inner.weight, init_score=P["li"] if offset else None)
    dh = lgb.Dataset(Xh, hold.target / hold.weight, weight=hold.weight, init_score=P["lh"] if offset else None,
                     reference=di)
    b = lgb.train(params, di, num_boost_round=5000, valid_sets=[dh],
                  callbacks=[lgb.early_stopping(patience, verbose=False)])
    inner_score = b.best_score["valid_0"]["tweedie"]
    n = max(int(b.best_iteration * 1.1), 1)
    Xt, cats = prep2(tr[feats]); Xv, _ = prep2(va[feats], cats)
    model = lgb.train(params, lgb.Dataset(Xt, tr.target / tr.weight, weight=tr.weight,
                                          init_score=P["lt"] if offset else None), num_boost_round=n)
    if offset:
        rate_tr = np.exp(P["lt"] + model.predict(Xt, raw_score=True))
        rate_va = np.exp(P["lv"] + model.predict(Xv, raw_score=True))
    else:
        rate_tr, rate_va = model.predict(Xt), model.predict(Xv)
    k = tr.target.sum() / (rate_tr * tr.weight).sum()
    return k * rate_va * va.weight.to_numpy(), dict(trees=n, inner_best=b.best_iteration,
                                                    inner_score=inner_score, k=k)


# ----------------------------------------------------- 2 and 3. method B and the grid
GRID = [dict(num_leaves=nl, min_data_in_leaf=md) for nl, md in product([15, 31, 63], [200, 1000, 3000])]
DEFAULT = dict(num_leaves=31, min_data_in_leaf=500)


def run_models(only_default):
    configs = [DEFAULT] if only_default else GRID
    fname = "2_method_b_default.csv" if only_default else "3_grid.csv"
    for line in LINES:
        df = data[line]["df"]
        feats = feats_of(df, line)
        for vy, tys in FOLDS:
            t0 = time.time()
            P = glm_parts(df, line, vy, tys, feats)
            va = P["va"]
            glm_pred = np.exp(P["lv"]) * va.weight.to_numpy()
            append_csv({"line": line, "year": vy, "model": "GLM v4", "config": "", **metrics(va, glm_pred)}, fname)
            for cfg, offset in product(configs, [False, True]):
                name = "method B" if offset else "GBM"
                try:
                    pred, info = fit_gbm(P, feats, dict(PARAMS2, **cfg), offset)
                    row = {"line": line, "year": vy, "model": name, "config": str(cfg), **info, **metrics(va, pred)}
                    append_csv(row, fname)
                    if not offset:
                        append_csv({"line": line, "year": vy, "model": "blend 50/50", "config": str(cfg),
                                    **metrics(va, 0.5 * glm_pred + 0.5 * pred)}, fname)
                    log(f"{fname} {line} {vy} {name} {cfg}: trees {info['trees']}, k {info['k']:.3f}, "
                        f"D2 {row['D2']:.4f}")
                except Exception:
                    log(f"FAILED {line} {vy} {name} {cfg}\n{traceback.format_exc()}")
            log(f"{fname} {line} {vy} done in {(time.time() - t0) / 60:.1f} min")


def summarise_grid():
    g = pd.read_csv(OUT / "3_grid.csv")
    rows = []
    for line in LINES:
        glm = g[(g.line == line) & (g.model == "GLM v4")].groupby("year").D2.mean()
        for name in ["GBM", "method B"]:
            s = g[(g.line == line) & (g.model == name)]
            pick = s.groupby("config").inner_score.mean().idxmin()           # chosen on training years only
            c = s[s.config == pick].set_index("year")
            rows.append({"line": line, "model": name, "chosen": pick, "trees": c.trees.round().tolist(),
                         "k": c.k.round(3).tolist(), "D2_avg": c.D2.mean(), "gain_vs_GLM": c.D2.mean() - glm.mean(),
                         "low": c.low.round(3).tolist(), "top": c.top.round(3).tolist()})
            if name == "GBM":
                b = g[(g.line == line) & (g.model == "blend 50/50") & (g.config == pick)].set_index("year")
                rows.append({"line": line, "model": "blend 50/50", "chosen": pick, "D2_avg": b.D2.mean(),
                             "gain_vs_GLM": b.D2.mean() - glm.mean(), "low": b.low.round(3).tolist(),
                             "top": b.top.round(3).tolist()})
        gl = g[(g.line == line) & (g.model == "GLM v4")].set_index("year")
        rows.append({"line": line, "model": "GLM v4", "D2_avg": gl.D2.mean(), "gain_vs_GLM": 0.0,
                     "low": gl.low.round(3).tolist(), "top": gl.top.round(3).tolist()})
    s = pd.DataFrame(rows)
    s.to_csv(OUT / "3_grid_summary.csv", index=False)
    log("3. grid summary (setting chosen on the held-out last training year):\n" + s.round(4).to_string(index=False))


from src.target import match_claims
log("=== start ===")
for step, fn in [("1 NCB check", ncb_check), ("2 method B (default settings)", lambda: run_models(True)),
                 ("3 settings grid", lambda: run_models(False)), ("3 summary", summarise_grid)]:
    try:
        log(f"--- {step} ---")
        fn()
    except Exception:
        log(f"STEP FAILED: {step}\n{traceback.format_exc()}")
log("=== all done ===")

12:55:31 === start ===
12:55:31 --- 1 NCB check ---
12:55:31 1. NCB check (2021-23):
  b  freq_per_1000  mean_gross  small_claim_share  exposure_share      peril
  0        140.630    2173.353              0.371           0.065 own_damage
1-2         92.219    2189.115              0.377           0.179 own_damage
3-4         67.116    2364.442              0.352           0.136 own_damage
5-8         50.514    2451.700              0.337           0.209 own_damage
 9+         41.684    2523.606              0.327           0.410 own_damage
  0        299.113    4268.036              0.310           0.065 all perils
1-2        207.186    4285.796              0.312           0.179 all perils
3-4        152.486    4305.879              0.318           0.136 all perils
5-8        118.675    3838.451              0.324           0.209 all perils
 9+        101.965    3924.684              0.326           0.410 all perils
12:55:31 --- 2 method B (default settings) ---
12:56:17 2_method_b_d

In [4]:
# 11d Monotone calibration of GBM / blend (home lines; motor for reference). Run in the 11 kernel after 11c.
# Problem: GBM and the blend over-price the lowest decile (A/E 0.56-0.74), which fails our tail rule.
# Fix tested: a monotone curve "predicted rate -> actual rate", fitted on the HELD-OUT LAST TRAINING YEAR with
# models trained on the earlier training years, then applied to the final model's predictions and rebalanced
# on the training years. The validation year is not used to build anything.
# Rule fixed before results: adopt the calibrated GBM or blend for a line if average validation D2 >= GLM v4
# + 0.003 and, in each year, lowest and highest decile A/E are no more than 0.1 further from 1 than GLM v4's.
from sklearn.isotonic import IsotonicRegression

CHOSEN = {"motor": dict(num_leaves=15, min_data_in_leaf=3000),       # chosen in 11c on the last training year
          "buildings": dict(num_leaves=15, min_data_in_leaf=1000),
          "contents": dict(num_leaves=15, min_data_in_leaf=3000)}


def gbm_hold_and_final(P, feats, params, patience=300):
    """GBM on inner years (trees by early stopping on the hold year) -> hold-year rates for calibration;
    then the final GBM on all training years -> training and validation rates (balanced)."""
    inner, hold, tr, va = P["inner"], P["hold"], P["tr"], P["va"]
    Xi, cats = prep2(inner[feats]); Xh, _ = prep2(hold[feats], cats)
    di = lgb.Dataset(Xi, inner.target / inner.weight, weight=inner.weight)
    dh = lgb.Dataset(Xh, hold.target / hold.weight, weight=hold.weight, reference=di)
    b = lgb.train(params, di, num_boost_round=5000, valid_sets=[dh],
                  callbacks=[lgb.early_stopping(patience, verbose=False)])
    k_in = inner.target.sum() / (b.predict(Xi, num_iteration=b.best_iteration) * inner.weight).sum()
    hold_rate = k_in * b.predict(Xh, num_iteration=b.best_iteration)
    n = max(int(b.best_iteration * 1.1), 1)
    Xt, cats = prep2(tr[feats]); Xv, _ = prep2(va[feats], cats)
    m = lgb.train(params, lgb.Dataset(Xt, tr.target / tr.weight, weight=tr.weight), num_boost_round=n)
    rt, rv = m.predict(Xt), m.predict(Xv)
    k = tr.target.sum() / (rt * tr.weight).sum()
    return hold_rate, k * rt, k * rv, n


def fit_calibration(pred_rate, actual_cost, weight, n_bins=50):
    """Monotone map of predicted rate -> actual rate, on weighted quantile bins (in log scale)."""
    lx = np.log(np.maximum(pred_rate, 1e-9))
    order = np.argsort(lx)
    cw = np.cumsum(weight[order]) / weight.sum()
    b = np.minimum((cw * n_bins).astype(int), n_bins - 1)
    df_ = pd.DataFrame({"b": b, "lx": lx[order], "w": weight[order], "a": actual_cost[order],
                        "p": pred_rate[order] * weight[order]})
    g = df_.groupby("b").apply(lambda s: pd.Series({"lx": np.average(s.lx, weights=s.w), "w": s.w.sum(),
                                                    "a_rate": s.a.sum() / s.w.sum()}))
    iso = IsotonicRegression(increasing=True, out_of_bounds="clip", y_min=1e-9)
    fitted = iso.fit_transform(g.lx, g.a_rate, sample_weight=g.w)
    xs, ys = g.lx.to_numpy(), np.log(np.maximum(fitted, 1e-9))
    return lambda rate: np.exp(np.interp(np.log(np.maximum(rate, 1e-9)), xs, ys))


def tails_ok(m, ref):
    return all(abs(m[c] - 1) <= abs(ref[c] - 1) + 0.1 for c in ["low", "top"])


rows = []
for line in LINES:
    df = data[line]["df"]
    feats = feats_of(df, line)
    params = dict(PARAMS2, **CHOSEN[line])
    for vy, tys in FOLDS:
        t0 = time.time()
        P = glm_parts(df, line, vy, tys, feats)
        tr, va, hold = P["tr"], P["va"], P["hold"]
        g_hold, g_tr, g_va = np.exp(P["lh"]), np.exp(P["lt"]), np.exp(P["lv"])
        b_hold, b_tr, b_va, n = gbm_hold_and_final(P, feats, params)
        cands = {"GLM v4": (None, g_tr, g_va),
                 "GBM": (b_hold, b_tr, b_va),
                 "blend": (0.5 * g_hold + 0.5 * b_hold, 0.5 * g_tr + 0.5 * b_tr, 0.5 * g_va + 0.5 * b_va)}
        out = {}
        for name, (h, rtr, rva) in cands.items():
            out[name] = rva * va.weight.to_numpy()
            if h is not None:
                f = fit_calibration(h, hold.target.to_numpy(), hold.weight.to_numpy())
                ctr, cva = f(rtr), f(rva)
                k2 = tr.target.sum() / (ctr * tr.weight).sum()                 # rebalance on training years
                out[name + " calibrated"] = k2 * cva * va.weight.to_numpy()
        for name, pred in out.items():
            rows.append({"line": line, "year": vy, "model": name, **metrics(va, pred)})
        log(f"11d {line} {vy}: GBM {n} trees, {time.time() - t0:.0f}s")

cal = pd.DataFrame(rows)
cal.to_csv(OUT / "4_calibration.csv", index=False)
for line in LINES:
    t = cal[cal.line == line].pivot(index="model", columns="year", values=["D2", "A/E", "low", "top"])
    t.columns = [f"{m}_{y}" for m, y in t.columns]
    t["D2_avg"] = t[["D2_2023", "D2_2024"]].mean(axis=1)
    print(f"\n===== {line} =====")
    print(t.sort_values("D2_avg", ascending=False).round(4).to_string())

print("\nrule: D2 >= GLM v4 + 0.003 and tails no more than 0.1 further from 1 than GLM v4 (each year)")
for line in LINES:
    c = cal[cal.line == line].set_index(["model", "year"])
    base = c.loc["GLM v4"]
    for m in ["GBM", "GBM calibrated", "blend", "blend calibrated"]:
        gain = c.loc[m].D2.mean() - base.D2.mean()
        tails = all(tails_ok(c.loc[(m, y)], base.loc[y]) for y in [2023, 2024])
        verdict = "ADOPT" if gain >= 0.003 and tails else "no"
        print(f"  {line:9s} {m:17s} gain {gain:+.4f} | tails {'ok' if tails else 'fail'} -> {verdict}")

15:25:23 11d motor 2023: GBM 259 trees, 93s
15:27:54 11d motor 2024: GBM 283 trees, 151s
15:28:41 11d buildings 2023: GBM 101 trees, 47s
15:30:01 11d buildings 2024: GBM 174 trees, 80s
15:30:49 11d contents 2023: GBM 108 trees, 49s
15:32:14 11d contents 2024: GBM 193 trees, 84s

===== motor =====
                  D2_2023  D2_2024  A/E_2023  A/E_2024  low_2023  low_2024  top_2023  top_2024  D2_avg
model                                                                                                 
blend              0.0543   0.0558    1.0565    1.0224    0.8857    0.9894    1.1139    1.0943  0.0551
blend calibrated   0.0529   0.0550    1.0628    1.0257    0.9236    1.1555    0.9868    1.0732  0.0540
GBM                0.0529   0.0544    1.0858    1.0434    0.9785    1.0473    1.1362    1.0858  0.0537
GBM calibrated     0.0519   0.0543    1.0900    1.0471    0.9780    1.2634    1.0496    1.0626  0.0531
GLM v4             0.0518   0.0543    1.0288    1.0022    0.9121    0.9795    1.0312

In [5]:
# 11e v5 = v4 with contents replaced by the GLM v4 + GBM 50/50 blend (passed the pre-set rule in 11d)
line = "contents"
df, loading = data[line]["df"], data[line]["loading"]
feats = feats_of(df, line)
glm = BandedTweedieGLM(**GLM_CONFIG[line]).fit(df[feats], df.target / df.weight, df.weight)
model, cats, k, n_trees = fit_gbm2(df, feats, params=dict(PARAMS2, **CHOSEN[line]))

p, pt = load_policies(line, "train"), load_policies(line, "test")
Ft = add_history(build_features(line, "test", policies=pt), pt, [p, pt], claims)
assert list(Ft.row_id) == list(pt.row_id)
Xt, _ = prep2(Ft[feats], cats)
rate = 0.5 * glm.predict_rate(Ft[feats]) + 0.5 * k * model.predict(Xt)
pred = np.maximum((rate + loading) * pt.exposure_clean.to_numpy(), 0)

v4 = pd.read_csv("../submissions/glm_v4.csv")
sub = v4.set_index("row_id")
old = sub.loc[Ft.row_id, "prediction"].to_numpy()
sub.loc[Ft.row_id, "prediction"] = pred
sub = sub.reset_index()
assert len(sub) == len(v4) and (sub.row_id == v4.row_id).all()
assert sub.prediction.notna().all() and (sub.prediction >= 0).all()
print(f"contents: {n_trees} trees, k {k:.3f}; contents total v5 / v4 = {pred.sum() / old.sum():.3f}; "
      f"rows changed by > 2x: {((pred / old > 2) | (pred / old < 0.5)).mean():.2%}")
sub.to_csv("../submissions/v5.csv", index=False)
print("wrote submissions/v5.csv")

contents: 269 trees, k 1.087; contents total v5 / v4 = 0.998; rows changed by > 2x: 0.02%
wrote submissions/v5.csv


In [ ]:
# 11f Claim history by peril group (counts only; previous year). Run in the 11 kernel (after 11, 11c).
# Rule fixed before results: adopt in the GLM for a line if average validation D2 rises by >= 0.0005 over
# GLM v4 and the A/E of rows with a past claim in the main group moves toward 1.
# Amounts are not used (valued at the data end, so they leak development after the cutoff).
from src.history import claim_history

GROUPS = {"motor": {"bi": ["bodily_injury"], "od": ["own_damage"], "tpp": ["third_party_prop"],
                    "oth": ["glass_hail", "theft"]},
          "buildings": {"wthr": ["flood", "storm", "subsidence"], "eow": ["escape_of_water"],
                        "oth": ["fire", "accidental_damage"]},
          "contents": {"theft": ["theft"], "eow": ["escape_of_water"], "oth": ["flood", "accidental_damage"]}}
MAIN = {"motor": "hist_bi_1y", "buildings": "hist_wthr_1y", "contents": "hist_theft_1y"}


def add_peril_history(F, p_rows, p_all, claims, line):
    out = F.copy()
    for g, perils in GROUPS[line].items():
        h = claim_history(p_rows, p_all, claims, years=(1,), perils=perils, prefix=f"hist_{g}")
        col = f"hist_{g}_1y"
        lvl = pd.Series(np.where(h[col] >= 1, "1+", "0"), index=h.index, dtype="object").where(h[col].notna())
        out = out.merge(pd.DataFrame({"row_id": h.row_id, col: lvl}), on="row_id", how="left", validate="one_to_one")
    assert len(out) == len(F)
    return out


PH = {}
for line in LINES:
    p, pt = load_policies(line, "train"), load_policies(line, "test")
    df = data[line]["df"]
    df2 = add_peril_history(df, p, [p, pt], claims, line)
    PH[line] = [f"hist_{g}_1y" for g in GROUPS[line]]
    data[line]["df_ph"] = df2
    print(f"{line}: share with >= 1 claim last year by group (2024): " +
          ", ".join(f"{c} {(df2[df2.year == 2024][c] == '1+').mean():.1%}" for c in PH[line]))

rows, ae = [], []
for line in LINES:
    df2 = data[line]["df_ph"]
    base = [c for c in df2.columns if c not in NON_FEATURES and c not in DROP[line] and c not in PH[line]]
    for vy, tys in FOLDS:
        t0 = time.time()
        tr, va = df2[df2.year.isin(tys)], df2[df2.year == vy]
        for name, feats in [("GLM v4", base), ("GLM v4 + peril history", base + PH[line])]:
            glm = BandedTweedieGLM(**GLM_CONFIG[line]).fit(tr[feats], tr.target / tr.weight, tr.weight)
            pred = glm.predict_rate(va[feats]) * va.weight.to_numpy()
            rows.append({"line": line, "year": vy, "model": name, **metrics(va, pred)})
            g = pd.DataFrame({"h": va[MAIN[line]].fillna("unknown").values, "a": va.target.values, "p": pred})
            s = g.groupby("h")[["a", "p"]].sum()
            for lvl, v in (s.a / s.p).items():
                ae.append({"line": line, "year": vy, "model": name, "main_group": lvl, "A/E": v})
        log(f"11f {line} {vy}: {time.time() - t0:.0f}s")

ph = pd.DataFrame(rows)
ph.to_csv(OUT / "5_peril_history.csv", index=False)
for line in LINES:
    t = ph[ph.line == line].pivot(index="model", columns="year", values=["D2", "A/E", "low", "top"])
    t.columns = [f"{m}_{y}" for m, y in t.columns]
    t["D2_avg"] = t[["D2_2023", "D2_2024"]].mean(axis=1)
    print(f"\n===== {line} =====")
    print(t.round(4).to_string())
print("\nA/E by previous-year claims in the main peril group (motor: BI; buildings: weather; contents: theft):")
print(pd.DataFrame(ae).pivot_table(index=["line", "model"], columns=["main_group", "year"], values="A/E")
      .round(3).to_string())
avg = ph.groupby(["line", "model"]).D2.mean().unstack()
print("\nrule: adopt if GLM D2 gain >= 0.0005 (and the main-group A/E moves toward 1):")
for line in LINES:
    d = avg.loc[line, "GLM v4 + peril history"] - avg.loc[line, "GLM v4"]
    print(f"  {line:9s} {d:+.4f}  {'adopt (check A/E table)' if d >= 0.0005 else 'do not adopt'}")